# Statistical Inference

## Introduction
### Hypothesis Tests
#### Step 1: Assumptions
#### Step 2: State the hypotheses and significance level
#### Step 3: Compute the test statistic
#### Step 4: Compute the $p$-value
#### Step 5: State your conclusion
### Confidence Intervals
## Comparing Means
### 2-sample Tests
#### Formal Set-up

In [ ]:
import pandas as pd
import numpy as np

from scipy import stats

import statsmodels.api as sm
from statsmodels.formula.api import ols
import statsmodels.stats.multicomp as mc

import seaborn as sns
import matplotlib.pyplot as plt
from itables import show

from ind5003 import inference

### Example: Abalone measurements t-test

In [ ]:
abl = pd.read_csv("data/abalone_sub.csv")
show(abl)

In [ ]:
abl = pd.read_csv("data/abalone_sub.csv")
abl.head()

In [ ]:
x = abl.viscera[abl.gender == "F"]
y = abl.viscera[abl.gender == "M"]

t_out = stats.ttest_ind(y, x)
ci_95 = t_out.confidence_interval()

print(f"""
The $p$-value for the test is {t_out.pvalue:.3f}. 
The actual value of the test statistic is {t_out.statistic:.3f}. 
The upper and lower limits of the CI are ({ci_95[0]:.3f}, {ci_95[1]:.3f}).
	""")

### Example: Abalone measurements histograms

In [ ]:
#| fig-align: center
#| label: fig-abalone-hist
#| fig-cap: Abalone dataset histograms
#| fig-pos: 'ht'
#| 
sns.displot(abl, x='viscera', col='gender', kind='hist', stat='density', 
            binwidth=0.1, height=2.5, aspect=1.2);

### Example: Abalone measurements QQ-plots

In [ ]:
#| fig-align: center
#| fig-pos: 'ht'
#| label: fig-abalone-qq
#| fig-cap: Abalone dataset qq-plots
#| 
f, axs = plt.subplots(1, 2, figsize=(8,3))
tmp = plt.subplot(121)
sm.qqplot(x, line="q", ax=tmp)
tmp.set_title('Females')
tmp = plt.subplot(122)
sm.qqplot(y, line="q", ax=tmp)
tmp.set_title('Males');

In [ ]:
abl.groupby('gender').describe()

## Paired Sample Tests
### Example: Reaction time of drivers
### Formal Set-up
### Example: Heart Rate Before/After Treadmill

In [ ]:
hr_df = pd.read_csv("data/health_promo_hr.csv")
p_test_out = stats.ttest_rel(hr_df.baseline, hr_df.after5)

print(f"""
The $p$-value for the test is {p_test_out.pvalue:.2g}. 
The difference in means is {hr_df.baseline.mean() - hr_df.after5.mean():.3f}. 
""")

In [ ]:
#| fig-align: center
#| fig-cap: "Agreement plot"
#| fig-pos: "ht"
#| label: fig-agreement

ax1 = hr_df.plot(x='baseline', y='after5',kind='scatter', marker='o', 
                 figsize=(4,3), edgecolor='blue', color='none')
group_means = hr_df.loc[:, ['baseline', 'after5']].mean(axis=0)
ax1.set_xlim(75, 105)
ax1.set_ylim(75,105)
ax1.plot([75,105], [75,105], color="lightblue", linestyle="dashed");
ax1.scatter(group_means.iloc[0], group_means.iloc[1],  marker='o', 
            edgecolor='blue', color='none', s=100);
ax1.set_title('Agreement of after5 and baseline');

## ANoVA
### Example: Tartar dataset
#### Tartar: Questions of Interest

In [ ]:
#| fig-pos: 'ht'
#| fig-align: 'center'
#| label: fig-tartar-bw
#| fig-cap: Boxplots for Tartar dataset

tartar = pd.read_table("data/tartar.txt", sep="\\s+")

plt.figure(figsize=(4, 3))
ax1 = sns.boxplot(tartar, x='treat', y='index')
ax1.set_xlabel("Treatment"); ax1.set_ylabel("Tartar Index");
ax1.set_title("Tartar on teeth");

In [ ]:
#| label: tbl-tartar-summary
#| tbl-cap: Tartar dataset summary

tartar.groupby('treat').describe()

### Formal Set-up
### $F$-Test in One-Way ANOVA
### Assumptions
### Example: Tartar ANOVA F-test

In [ ]:
tartar_lm = ols("index ~ C(treat, Treatment('Control'))", data=tartar).fit()
anova_tab = sm.stats.anova_lm(tartar_lm, type=3,)
anova_tab

In [ ]:
tartar_lm.summary()

In [ ]:
#| fig-align: center
#| label: fig-tartar-norm
#| fig-cap: Normality plots for Tartar dataset

inference.check_normality(pd.Series(tartar_lm.resid_pearson))

### Comparing specific groups {#sec-spec-grps}
### Example: $HMP$ vs. $P_2 O_7$

In [ ]:
est1  = tartar_lm.params.iloc[2] - tartar_lm.params.iloc[1]
MSW = tartar_lm.mse_resid
df = tartar_lm.df_resid
q1 = -stats.t.ppf(0.025, df)

lower_ci = est1 - q1*np.sqrt(MSW * (1/8 + 1/9))
upper_ci = est1 + q1*np.sqrt(MSW * (1/8 + 1/9))
print("The 95% CI for the diff. between HMP and P2O7 is" +
      f"({lower_ci:.3f}, {upper_ci:.3f}).") 

### Contrast Estimation
### Example: Tartar: Comparing collections of groups

In [ ]:
c1 = np.array([-1, 0.5, 0.5])
n_vals = np.array([9, 8, 9,])
L = np.sum(c1 * np.append(0, tartar_lm.params.iloc[1:].to_numpy()))

MSW = tartar_lm.mse_resid
df = tartar_lm.df_resid
q1 = -stats.t.ppf(0.025, df)
se1 = np.sqrt(MSW*np.sum(c1**2 / n_vals))

lower_ci = L - q1*se1
upper_ci = L + q1*se1
print("The 95% CI for the diff. between the two groups is " +
      f"({lower_ci:.3f}, {upper_ci:.3f}).") 

### Multiple Comparisons
#### Bonferroni
#### TukeyHSD
### Example: Multiple comparisons

In [ ]:
cp = mc.MultiComparison(tartar.index, tartar.treat)
tk = cp.tukeyhsd()
print(tk)

In [ ]:
#| fig-align: center
#| fig-pos: 'ht'
#| fig-cap: Multiple comparisons plot, tartar dataset
#| label: fig-tartar-hsd
tk.plot_simultaneous(comparison_name = 'Control', figsize=(6,3));

## Categorical Variables
### Example: Chest pain and gender

In [ ]:
#| fig-align: center
#| fig-pos: 'ht'
#| fig-cap: Contingency table for chest pain and gender
#| label: fig-chest-tbl

chest_array = np.array([[46, 474], [37, 516]])
#print(chest_array)
plt.figure(figsize=(4, 2)) 
sns.heatmap(chest_array, annot=True, square=True, fmt='', 
            xticklabels=['pain', 'no pain'],
            yticklabels=['Male', 'Female'], 
            cmap='Reds', cbar=False, );

### $\chi^2$-Test for Independence 
### Example: Chest pain and gender expected counts

In [ ]:
col_prop,row_prop = sm.stats.Table(chest_array).marginal_probabilities

print(f"""
  The proportion of males in the sample was {col_prop[0]:.3f}.
  The proportion of all patients who experienced chest pain was {row_prop[0]:.3f}.
  """)

### Example: Chest pain and gender $\chi^2$ test

In [ ]:
chisq_output = stats.chi2_contingency(chest_array, correction=False)

print(f"""
  The p-value is {chisq_output.pvalue:.3f}.
  The test-statistic value is {chisq_output.statistic:.3f}.""")

In [ ]:
chisq_output.expected_freq

### Example: Chest pain and gender Fisher Exact test

In [ ]:
stats.fisher_exact(chest_array)

### Measures of Association
### Odds Ratio
### Example: Chest pain and gender odds ratio

In [ ]:
chest_tab2 = sm.stats.Table2x2(chest_array)

chest_tab2.summary()

### For Ordinal Variables
### Example: Job satisfaction by income

In [ ]:
#| fig-align: center
#| fig-pos: 'ht'
#| fig-cap: Contingency table for job satisfaction by income
#| label: fig-job-income-tbl


us_svy_tab = np.array([[1, 3, 10, 6], 
                      [2, 3, 10, 7],
                      [1, 6, 14, 12],
                      [0, 1,  9, 11]])
col_names = ['V. Diss', 'L. Diss', 'M. Sat', 'V. Sat']
row_names = ['<15K', '15-25K', '25-40K', '>40K']

plt.figure(figsize=(4, 2)) 
sns.heatmap(us_svy_tab, annot=True, square=True, fmt='', 
            xticklabels=col_names,
            yticklabels=row_names, 
            cmap='Reds', cbar=False, );

In [ ]:
dim1 = us_svy_tab.shape
x = []; y=[]
for i in range(0, dim1[0]):
    for j in range(0, dim1[1]):
        for k in range(0, us_svy_tab[i,j]):
            x.append(i)
            y.append(j)

In [ ]:
stats.kendalltau(x, y, variant='b')

## Summary 
## References {#sec-02-ref}
### Website References
### Documentation links
## Exercises